In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [4]:
import pandas as pd
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.to_csv("Titanic-Dataset.csv", index=False)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [7]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [8]:
df["Age"] = df["Age"].fillna(df["Age"].median())

df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

In [9]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64

In [10]:
df = df.drop(["Cabin", "PassengerId", "Name", "Ticket"], axis=1, errors="ignore")

In [11]:
df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)

In [12]:
print(df.columns)

Index(['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male',
       'Embarked_Q', 'Embarked_S'],
      dtype='object')


In [13]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

In [14]:
df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape)

(712, 8)
(179, 8)


In [16]:
rf = RandomForestClassifier(random_state=42)
param_grid = {"n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]}

In [17]:
grid_search = GridSearchCV(estimator=rf,param_grid=param_grid,cv=5,scoring="accuracy",n_jobs=-1)

grid_search.fit(X_train, y_train)
print("Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


In [18]:
print("Best Cross-Validation Accuracy:")
print(grid_search.best_score_)

Best Cross-Validation Accuracy:
0.8356446370530877


In [19]:
best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.8156424581005587


In [20]:
svm = SVC(random_state=42)
param_dist = {"C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]}

In [21]:
random_search = RandomizedSearchCV(
    estimator=svm,
    param_distributions=param_dist,
    n_iter=6,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1)
random_search.fit(X_train, y_train)


,estimator,SVC(random_state=42)
,param_distributions,"{'C': [0.1, 1, ...], 'gamma': ['scale', 'auto'], 'kernel': ['linear', 'rbf']}"
,n_iter,6
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [22]:
print("Best Parameters:")
print(random_search.best_params_)

print("Best Cross-Validation Accuracy:")
print(random_search.best_score_)

Best Parameters:
{'kernel': 'linear', 'gamma': 'auto', 'C': 100}
Best Cross-Validation Accuracy:
0.7892938047867626


In [23]:
best_svm = random_search.best_estimator_
y_pred_svm = best_svm.predict(X_test)

svm_accuracy = accuracy_score(y_test, y_pred_svm)
print("SVM Test Accuracy:", svm_accuracy)

SVM Test Accuracy: 0.776536312849162


In [24]:
results = pd.DataFrame({
    "Model": ["Random Forest", "SVM"],
    "Search Method": ["Grid Search", "Randomized Search"],
    "Best Parameters": [
        str(grid_search.best_params_),
        str(random_search.best_params_)
    ],
    "CV Accuracy": [
        grid_search.best_score_,
        random_search.best_score_
    ],
    "Test Accuracy": [
        accuracy,
        svm_accuracy
    ]
})

results

,Model,Search Method,Best Parameters,CV Accuracy,Test Accuracy
0,Random Forest,Grid Search,"{'max_depth': 5, 'min_samples_leaf': 1, 'min_s...",0.835645,0.815642
1,SVM,Randomized Search,"{'kernel': 'linear', 'gamma': 'auto', 'C': 100}",0.789294,0.776536


In [25]:
results.to_csv("experiment_table.csv", index=False)

In [26]:
if accuracy > svm_accuracy:
    print("Best Model: Random Forest")
else:
    print("Best Model: SVM")

Best Model: Random Forest


# Bayesian Search

Bayesian Search is an advanced hyperparameter optimization technique that uses the results of previous evaluations to choose the next set of hyperparameters. Unlike Grid Search, which tries every combination, and Randomized Search, which selects random combinations, Bayesian Search intelligently explores the search space. This often finds good hyperparameters with fewer model evaluations, making it efficient for computationally expensive models.

# Tuning Limits

Hyperparameter tuning should use a reasonable search space. A very large search space increases computation time, while a very small search space may miss the optimal parameters. Grid Search is exhaustive but slower, Randomized Search is faster and more efficient for large search spaces, and Bayesian Search is suitable when training models is computationally expensive.